In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import tensorflow as tf
import subprocess
import os

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            # Enable memory growth so it doesn't seize the whole card
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

In [3]:
detector = YOLO("yolo26s.pt") 
classifier = tf.keras.models.load_model('traffic_light_cnn.h5')

In [4]:
# Configuration
CLASS_NAMES = ['Green', 'Red', 'Yellow'] # Ensure this matches your training order
COLOR_MAP = {'Green': (0, 255, 0), 'Red': (0, 0, 255), 'Yellow': (0, 255, 255)}

In [5]:
name ='27feb_crv'
input_name = 'video/'+name+'.mp4'
base_output = f'/home/yashas/Desktop/traffic_light/traffic_analysis_output_{name}'
extension = '.mp4'
# output_name = 'output_video/traffic_analysis_output_'+ name+'.mp4'
output_path = f"{base_output}{extension}"
counter = 1

# Check if the file exists and increment a counter until a unique name is found
while os.path.exists(output_path):
    output_path = f"{base_output}_{counter}{extension}"
    counter += 1

cap = cv2.VideoCapture(0)
# output_path = output_name

In [6]:
# 2. Get Video Properties for Saving
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# SET CUSTOM FPS
# output_fps = 2  # This makes 2 frames last 1 second

In [7]:
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

In [ ]:
# Setup FFmpeg pipe for GPU encoding (NVENC)
# This sends raw frames from Python directly to the GPU encoder
# cmd = [
#     # 'ffmpeg',
#     '/home/yashas/Downloads/ffmpeg-7.0.2-amd64-static/ffmpeg',
#     '-y',                                # Overwrite output
#     '-f', 'rawvideo',                    # Input is raw pixels from Python
#     '-vcodec', 'rawvideo',
#     '-s', f'{width}x{height}',           # Frame size
#     '-pix_fmt', 'bgr24',                 # OpenCV default format
#     '-r', str(fps),                      # Framerate
#     '-i', '-',                           # Read from stdin (the pipe)
#     '-c:v', 'h264_nvenc',                # <--- THE GPU ENCODER
#     # '-c:v', 'libx264',
#     # --- QUALITY TWEAKS START HERE ---
#     '-preset', 'p7',             # p6 or p7 for high quality (slower but cleaner)
#     # '-preset', 'fast',
#     '-rc', 'vbr',                # Variable Bitrate mode
#     '-cq', '23',                 # The "CRF" equivalent for NVENC (18-23 is the sweet spot)
#     '-b:v', '0',                 # Required when using -cq to let it vary freely
#     '-profile:v', 'high',        # Use High profile for better compression
#     '-spatial-aq', '1',          # Spatial Adaptive Quantization (helps with grain/textures)
#     '-temporal-aq', '1',         # Temporal Adaptive Quantization
#     # '-profile:v', 'high',
#     # ---------------------------------
#         '-pix_fmt', 'yuv420p',               # Format for maximum compatibility
#     output_path
# ]

cmd = [
    "/home/yashas/Downloads/ffmpeg-7.0.2-amd64-static/ffmpeg",  # point to your extracted binary
    "-y",
    "-f", "rawvideo",
    "-vcodec", "rawvideo",
    "-s", f"{width}x{height}",
    "-pix_fmt", "bgr24",
    "-r", str(fps),
    "-i", "-",
    "-c:v", "libx264",       # CPU-safe
    "-preset", "fast",
    "-pix_fmt", "yuv420p",
    output_path
]


In [14]:
proc = subprocess.Popen(
    cmd,
    stdin=subprocess.PIPE,
    stderr=subprocess.PIPE  # <-- capture FFmpeg errors
)


FileNotFoundError: [Errno 2] No such file or directory: '/home/yashas/Downloads/ffmpeg-7.0.2-static/ffmpeg'

In [10]:
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
        
    frame_id = cap.get(cv2.CAP_PROP_POS_FRAMES)
    
    # if frame_id % (fps // 2) != 0:
    #     continue # Skip frames to only process 2 per "real" second

    img_h, img_w, _ = frame.shape
    img_center_x = img_w / 2

    # 3. Detection (Class 9 is traffic light)
    results = detector(frame, verbose=False, classes=[9])[0]
    
    lights = results.boxes
    if len(lights) > 0:
       

        best_box = max(lights, key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0]) * (b.xyxy[0][3]-b.xyxy[0][1]))
        x1, y1, x2, y2 = map(int, best_box.xyxy[0])

        
            
        # 5. Crop and Classify with your Vertical CNN
        # Note: Ensure crop is not empty
        crop = frame[max(0, y1):y2, max(0, x1):x2]
        if crop.size > 0:
            # Resize to your new vertical shape (64 width, 128 height)
            crop_resized = cv2.resize(crop, (64, 64)) / 255.0
            prediction = classifier.predict(np.expand_dims(crop_resized, axis=0), verbose=0)
            
            color_idx = np.argmax(prediction)
            color_label = CLASS_NAMES[color_idx]
            conf = prediction[0][color_idx]
            
            # 6. Draw UI
            bgr_color = COLOR_MAP.get(color_label, (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
            
            label_text = f"{color_label} {conf:.2%}"
            cv2.putText(frame, label_text, (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, bgr_color, 2)
            print("Signal_detected")

    # # 5. Save the frame to file
    # out.write(frame)
    try:
        proc.stdin.write(frame.tobytes())
    except BrokenPipeError:
        # Read FFmpeg stderr to see why it crashed
        err = proc.stderr.read().decode()
        print("FFmpeg crashed:\n", err)
        break
    # proc.stdin.write(frame.tobytes())
    
    # 7. Fast Rendering
    # cv2.imshow("YOLOv10 + Vertical CNN Traffic Monitor", frame)

    # Press 'q' to exit
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break

cap.release()
proc.stdin.close()
proc.wait()
cv2.destroyAllWindows()
print(f"Video saved successfully to {output_path}")

FFmpeg crashed:
 ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100